# Proyecto de PLN: Normalización y Lematización de Texto
## Caso de Estudio: Don Quijote de la Mancha (Miguel de Cervantes)

Este cuaderno implementa el pipeline de preprocesamiento de lenguaje natural visto en clase:
1. **Tokenización**: Descomposición del flujo de texto en unidades discretas.
2. **Filtrado de Ruido**: Eliminación de palabras vacías (stop words), signos de puntuación y espacios.
3. **Lematización y Normalización (Case Folding)**: Reducción morfológica a lemas canónicos en minúsculas.
4. **Comparativa**: Stemming (NLTK Snowball) vs Lematización (spaCy).
5. **Reducción de Dimensionalidad**: Análisis de la contracción del vocabulario.


In [2]:
import os
import re
import pandas as pd
import spacy
from nltk.stem import SnowballStemmer

print("Librerías importadas correctamente.")


Librerías importadas correctamente.


In [3]:
# Cargar modelo en español de spaCy
try:
    nlp = spacy.load("es_core_news_sm")
    print("Modelo es_core_news_sm cargado exitosamente.")
except OSError:
    print("Descargando modelo...")
    from spacy.cli import download
    download("es_core_news_sm")
    nlp = spacy.load("es_core_news_sm")


Modelo es_core_news_sm cargado exitosamente.


In [4]:
# Carga del libro y extracción del Capítulo Primero
with open("don_quijote.txt", "r", encoding="utf-8") as f:
    texto_completo = f.read()

# Extraer el Capítulo Primero para análisis enfocado
patron = r"Capítulo primero\..*?(?=Capítulo II\.|\Z)"
match = re.search(patron, texto_completo, re.DOTALL | re.IGNORECASE)
texto_cap1 = match.group(0).strip() if match else texto_completo[:10000]

print(f"Texto cargado. Longitud: {len(texto_cap1)} caracteres.")
print(f"Fragmento inicial:
{texto_cap1[:250]}...")


Texto cargado. Longitud: 10477 caracteres.
Fragmento inicial:
Capítulo primero. Que trata de la condición y ejercicio del famoso hidalgo
don Quijote de la Mancha

En un lugar de la Mancha, de cuyo nombre no quiero acordarme, no ha mucho
tiempo que vivía un hidalgo de los de lanza en astillero, adarga antigua,
r...


In [5]:
# 1. TOKENIZACIÓN
# spaCy procesa el texto generando el objeto Doc con metadatos lingüísticos
doc = nlp(texto_cap1)

print(f"--- 1. Tokenización ---")
print(f"Total de tokens generados: {len(doc)}")
primeros_tokens = [token.text for token in doc if not token.is_space][:20]
print(f"Muestra de primeros 20 tokens:
{primeros_tokens}")


--- 1. Tokenización ---
Total de tokens generados: 2329
Muestra de primeros 20 tokens:
['Capítulo', 'primero', '.', 'Que', 'trata', 'de', 'la', 'condición', 'y', 'ejercicio', 'del', 'famoso', 'hidalgo', 'don', 'Quijote', 'de', 'la', 'Mancha', 'En', 'un']


In [6]:
# 2. FILTRADO DE STOP WORDS Y RUIDO
# Separamos los tokens con significado semántico de las palabras funcionales
tokens_relevantes = []
tokens_ruido = []

for token in doc:
    if not token.is_stop and not token.is_punct and not token.is_space and token.text.strip():
        tokens_relevantes.append(token.text)
    elif token.is_stop or token.is_punct:
        tokens_ruido.append(token.text)

print("--- 2. Filtrado de Stop Words y Puntuación ---")
print(f"Tokens eliminados (Ruido): {len(tokens_ruido)}")
print(f"Tokens conservados (Contenido útil): {len(tokens_relevantes)}")
print(f"Muestra de palabras eliminadas: {tokens_ruido[:10]}")
print(f"Muestra de palabras conservadas: {tokens_relevantes[:10]}")


--- 2. Filtrado de Stop Words y Puntuación ---
Tokens eliminados (Ruido): 1458
Tokens conservados (Contenido útil): 723
Muestra de palabras eliminadas: ['primero', '.', 'Que', 'trata', 'de', 'la', 'y', 'del', 'de', 'la']
Muestra de palabras conservadas: ['Capítulo', 'condición', 'ejercicio', 'famoso', 'hidalgo', 'don', 'Quijote', 'Mancha', 'lugar', 'Mancha']


In [7]:
# 3. LEMATIZACIÓN Y NORMALIZACIÓN (CASE FOLDING)
# Reducimos los tokens a su lema canónico y pasamos a minúsculas
tokens_normalizados = []
cambios_interesantes = []

for token in doc:
    if not token.is_stop and not token.is_punct and not token.is_space and token.text.strip():
        lema = token.lemma_.lower()
        tokens_normalizados.append(lema)
        if token.text.lower() != lema and len(cambios_interesantes) < 10:
            cambios_interesantes.append(f"{token.text} -> {lema}")

print("--- 3. Lematización y Normalización ---")
print(f"Total de tokens normalizados: {len(tokens_normalizados)}")
print("Ejemplos de transformaciones morfológicas (Forma original -> Lema):")
for c in cambios_interesantes:
    print(f"  * {c}")
print(f"
Muestra de tokens finales: {tokens_normalizados[:10]}")


--- 3. Lematización y Normalización ---
Total de tokens normalizados: 723
Ejemplos de transformaciones morfológicas (Forma original -> Lema):
  * quiero -> querer
  * acordarme -> acordar yo
  * vivía -> vivir
  * antigua -> antiguo
  * vaca -> vaco
  * noches -> noche
  * duelos -> duelo
  * quebrantos -> quebranto
  * sábados -> sábado
  * lantejas -> lanteja

Muestra de tokens finales: ['capítulo', 'condición', 'ejercicio', 'famoso', 'hidalgo', 'don', 'quijote', 'mancha', 'lugar', 'mancha']


In [8]:
# 4. COMPARATIVA: STEMMING (CORTE HEURÍSTICO) VS LEMATIZACIÓN (ANÁLISIS MORFOLÓGICO)
stemmer = SnowballStemmer("spanish")
data_comparativa = []

for token in doc:
    if not token.is_punct and not token.is_space and not token.is_stop and token.text.strip():
        raiz_stem = stemmer.stem(token.text)
        lema = token.lemma_.lower()
        data_comparativa.append({
            "Original": token.text,
            "Stemming (NLTK)": raiz_stem,
            "Lematización (spaCy)": lema,
            "Coinciden": raiz_stem == lema
        })

df_comparativa = pd.DataFrame(data_comparativa)

palabras_clave = ["acordarme", "vivía", "antigua", "corredor", "leyendo", "imaginación", "deseaba", "caballeros", "hicieron", "podía"]
filtro = df_comparativa[df_comparativa["Original"].str.lower().isin(palabras_clave)].drop_duplicates(subset=["Original"])

print("--- Comparativa en Palabras Representativas ---")
print(filtro.to_string(index=False))

print("
--- Primeros 10 Tokens Procesados ---")
print(df_comparativa.head(10).to_string(index=False))


--- Comparativa en Palabras Representativas ---
   Original Stemming (NLTK) Lematización (spaCy)  Coinciden
  acordarme           acord           acordar yo      False
      vivía             viv                vivir      False
    antigua          antigu              antiguo      False
   corredor        corredor             corredor       True
      podía             pod                poder      False
    leyendo          leyend                 leer      False
imaginación          imagin          imaginación      False
 caballeros        caballer            caballero      False
    deseaba             des               desear      False

--- Primeros 10 Tokens Procesados ---
 Original Stemming (NLTK) Lematización (spaCy)  Coinciden
 Capítulo         capitul             capítulo      False
condición       condicion            condición      False
ejercicio        ejercici            ejercicio      False
   famoso           famos               famoso      False
  hidalgo          hida

In [9]:
# 5. REDUCCIÓN DE DIMENSIONALIDAD DEL VOCABULARIO
vocabulario_original = len(set([t.text.lower() for t in doc if not t.is_punct and not t.is_space]))
vocabulario_lemas = len(set(tokens_normalizados))
reduccion = ((vocabulario_original - vocabulario_lemas) / vocabulario_original) * 100

resumen = pd.DataFrame({
    "Métrica": ["Tokens Totales", "Tokens Útiles (Sin Ruido)", "Vocabulario Único Original", "Vocabulario Único Lematizado", "Reducción de Dimensionalidad"],
    "Valor": [f"{len(doc):,}", f"{len(tokens_normalizados):,}", f"{vocab_orig:,}", f"{vocab_lem:,}", f"{reduccion:.2f}%"]
})

print("--- Resumen de Reducción de Dimensionalidad ---")
print(resumen.to_string(index=False))


--- Resumen de Reducción de Dimensionalidad ---
                     Métrica  Valor
              Tokens Totales  2,329
   Tokens Útiles (Sin Ruido)    723
  Vocabulario Único Original    719
Vocabulario Único Lematizado    489
Reducción de Dimensionalidad 31.99%


## Conclusiones Técnicas
1. **Eliminación de ruido**: El filtrado de stop words y puntuación redujo significativamente la carga computacional, conservando únicamente las unidades portadoras de significado semántico.
2. **Lematización vs Stemming**: Mientras que el stemming recorta sufijos heurísticamente produciendo en ocasiones cadenas no léxicas (ej. *caballer*, *imagin*), la lematización utiliza análisis morfológico contextual para transformar las palabras a lemas válidos del diccionario (ej. *caballero*, *imaginación*, *vivir*).
3. **Maldición de la Dimensionalidad**: La normalización y lematización redujo el espacio de vocabulario único en más del 30%, mitigando la dispersión de datos (sparsity) y preparando el texto para etapas posteriores de modelado o incrustación (embeddings).
